# Module 6 - Deploy and Monitor

In this notebook you deploy the travel assistant as an AWS Lambda function, then review its model calls, latency by service tier, and cost per feature.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [Package the assistant](#s1)
4. [Deploy to AWS Lambda](#s2)
5. [Invoke the deployed assistant](#s3)
6. [Review invocation logs](#s4)
7. [Review service tier metrics](#s5)
8. [Attribute cost with request metadata](#s6)
9. [Try it](#try-it)
10. [Conclusion and key takeaways](#conclusion)
11. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

Over Modules 1-5 you built the two parts of AnyCompany's travel assistant in notebooks:

| Request | What it does | Model and tier | Built in |
|:--|:--|:--|:--|
| `ask` | Answers a travel question from the policy, with citations | Claude Sonnet 5, Standard | Modules 1, 3, 4 |
| `check_expense` | Checks an expense report and returns approved or flagged, with a reason | gpt-oss-120b, Priority | Modules 2, 4, 5 |

In this module you deploy both requests as one [AWS Lambda](https://docs.aws.amazon.com/lambda/latest/dg/welcome.html) function, then use the invocation logs from Module 0 and CloudWatch metrics to see what it calls, how fast it is, and what each feature costs.

## 2. Setup <a id="setup"></a>

Run the cell below to create the clients and find the knowledge base from Module 3 and the guardrail from Module 4. The deployed function uses them both.

In [ ]:
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

lambda_client = boto3.client("lambda", region_name=REGION)
iam = boto3.client("iam")
logs = boto3.client("logs", region_name=REGION)
cloudwatch = boto3.client("cloudwatch", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)

# The models the deployed function calls (set in app/lambda_function.py). "global." is a
# global cross-Region inference profile; gpt-oss-120b is called directly in this Region.
CHAT_MODEL_ID = "global.anthropic.claude-sonnet-5"
EXPENSE_MODEL_ID = "openai.gpt-oss-120b-1:0"

FUNCTION_NAME = "aim317-travel-assistant"
LAMBDA_ROLE_NAME = "aim317-lambda-role"
LOG_GROUP_NAME = "/aim317/bedrock-invocations"


def find_knowledge_base(name):
    """Return the ID of the knowledge base with this name, or None."""
    for kb in bedrock_agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"]:
        if kb["name"] == name:
            return kb["knowledgeBaseId"]
    return None


def find_guardrail(name):
    """Return the ID and latest published version of the guardrail with this name, or (None, None)."""
    for guardrail in bedrock.list_guardrails(maxResults=100)["guardrails"]:
        if guardrail["name"] == name:
            versions = [
                g["version"] for g in bedrock.list_guardrails(guardrailIdentifier=guardrail["id"])["guardrails"]
                if g["version"] != "DRAFT"
            ]
            return guardrail["id"], max(versions, key=int) if versions else None
    return None, None


KNOWLEDGE_BASE_ID = find_knowledge_base("aim317-travel-policy") or find_knowledge_base("aim317-travel-policy-prebuilt")
GUARDRAIL_ID, GUARDRAIL_VERSION = find_guardrail("aim317-travel-guardrail")

if KNOWLEDGE_BASE_ID is None:
    print("No knowledge base found. Run Module 3 - Grounding with a Managed Knowledge Base first.")
elif GUARDRAIL_VERSION is None:
    print("No published guardrail version found. Run Module 4 - Guardrails first.")
else:
    print(f"Region:          {REGION}")
    print(f"Knowledge base:  {KNOWLEDGE_BASE_ID}")
    print(f"Guardrail:       {GUARDRAIL_ID}, version {GUARDRAIL_VERSION}")

## 3. Package the assistant <a id="s1"></a>

The function's code is in [`app/lambda_function.py`](../app/lambda_function.py). It contains the same steps you ran in the notebooks, with one entry point, `lambda_handler`, that routes each request:

```
ask:
  1. Retrieve policy passages from the knowledge base
  2. Converse on Claude Sonnet 5 with the guardrail and requestMetadata tags
  3. Return the answer, citations, whether the guardrail intervened, and the updated history

check_expense:
  1. Retrieve the policy passages for each expense type in the report
  2. Converse on gpt-oss-120b, serviceTier priority, guardrail attached
  3. Return {"status": "approved" | "flagged", "reasons": [...], "policy_refs": [...]}
```

The function keeps no state: for a follow-up question, the caller sends the history from the previous answer. The knowledge base and guardrail IDs come from environment variables.

Run the cell below to read the start of the file.

In [ ]:
with open("../app/lambda_function.py") as f:
    print(f.read()[:1500])

Managed Knowledge Base retrieval needs boto3 1.43.98 or later, so the package includes that version rather than the one in the [Lambda Python runtime](https://docs.aws.amazon.com/lambda/latest/dg/lambda-python.html).

Run the cell below to build the deployment package: the function code and boto3. It takes up to a minute.

In [ ]:
build_dir = tempfile.mkdtemp()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--target", build_dir, "boto3>=1.43.98"],
    check=True,
)
shutil.copy("../app/lambda_function.py", build_dir)

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as package:
    for folder, _, files in os.walk(build_dir):
        for name in files:
            path = os.path.join(folder, name)
            package.write(path, os.path.relpath(path, build_dir))
DEPLOYMENT_PACKAGE = zip_buffer.getvalue()
shutil.rmtree(build_dir)

print(f"Deployment package: {len(DEPLOYMENT_PACKAGE) / 1_000_000:.1f} MB")

## 4. Deploy to AWS Lambda <a id="s2"></a>

The function runs with an IAM role that allows only what the code does: invoking the two models, retrieving from the knowledge base, applying the guardrail, and writing its own logs.

Run the cell below to get or create the role.

In [ ]:
def get_or_create_lambda_role():
    """Return the ARN of the function's role, creating the role if needed."""
    try:
        return iam.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
    except iam.exceptions.NoSuchEntityException:
        pass

    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }],
    }
    permissions = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": "bedrock:InvokeModel",
                "Resource": [
                    # A global inference profile can route to the model in any Region.
                    "arn:aws:bedrock:*::foundation-model/anthropic.claude-sonnet-5",
                    f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:inference-profile/global.anthropic.claude-sonnet-5",
                    f"arn:aws:bedrock:{REGION}::foundation-model/openai.gpt-oss-120b-1:0",
                ],
            },
            {
                "Effect": "Allow",
                "Action": "bedrock:Retrieve",
                "Resource": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/*",
            },
            {
                "Effect": "Allow",
                "Action": "bedrock:ApplyGuardrail",
                "Resource": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:guardrail/*",
            },
        ],
    }
    role = iam.create_role(RoleName=LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy),
                           Description="AIM317 workshop: the travel assistant Lambda function")
    iam.attach_role_policy(RoleName=LAMBDA_ROLE_NAME,
                           PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
    iam.put_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyName="travel-assistant-bedrock",
                        PolicyDocument=json.dumps(permissions))
    return role["Role"]["Arn"]


LAMBDA_ROLE_ARN = get_or_create_lambda_role()
print("Function role:", LAMBDA_ROLE_ARN)

Run the cell below to create or update the function. The environment variables tell the code which knowledge base and guardrail version to use.

In [ ]:
FUNCTION_SETTINGS = {
    "Runtime": "python3.13",
    "Handler": "lambda_function.lambda_handler",
    "Role": LAMBDA_ROLE_ARN,
    "Timeout": 60,
    "MemorySize": 512,
    "Environment": {"Variables": {
        "KNOWLEDGE_BASE_ID": KNOWLEDGE_BASE_ID,
        "GUARDRAIL_ID": GUARDRAIL_ID,
        "GUARDRAIL_VERSION": GUARDRAIL_VERSION,
    }},
}

try:
    lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=DEPLOYMENT_PACKAGE)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)
    lambda_client.update_function_configuration(FunctionName=FUNCTION_NAME, **FUNCTION_SETTINGS)
    print(f"Updated function {FUNCTION_NAME}")
except lambda_client.exceptions.ResourceNotFoundException:
    for attempt in range(10):
        try:
            lambda_client.create_function(FunctionName=FUNCTION_NAME, Code={"ZipFile": DEPLOYMENT_PACKAGE},
                                          Description="AnyCompany travel assistant", **FUNCTION_SETTINGS)
            break
        except lambda_client.exceptions.InvalidParameterValueException:
            print("Waiting for the role to be ready...")
            time.sleep(10)
    print(f"Created function {FUNCTION_NAME}")

lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)
print("State:", lambda_client.get_function(FunctionName=FUNCTION_NAME)["Configuration"]["State"])

## 5. Invoke the deployed assistant <a id="s3"></a>

An application would call the function with an event describing the request. Here you call it from the notebook with [`lambda.invoke`](https://docs.aws.amazon.com/lambda/latest/api/API_Invoke.html).

Run the cell below to define a small helper and ask a travel question.

In [ ]:
INVOKE_STARTED = time.time()  # Used in section 6 to find the log records of these calls.


def invoke(event):
    """Invoke the deployed assistant and return its response."""
    response = lambda_client.invoke(FunctionName=FUNCTION_NAME, Payload=json.dumps(event))
    payload = json.loads(response["Payload"].read())
    if "FunctionError" in response:
        raise RuntimeError(f"The function failed: {payload.get('errorMessage')}")
    return payload


started = time.time()
result = invoke({"action": "ask", "question": "What is AnyCompany's nightly hotel limit in Chicago?"})
print(f"({time.time() - started:.1f} s)\n")
print(result["answer"])
print("Citations:", [c["source"] for c in result["citations"]])
print("Guardrail intervened:", result["guardrail_intervened"])

The first call after a deployment takes longer, because Lambda initializes a new execution environment (a cold start).

To continue the conversation, send the `history` returned by the previous answer. Run the cell below to ask a follow-up question.

In [ ]:
started = time.time()
follow_up = invoke({
    "action": "ask",
    "question": "What about New York?",
    "history": result["history"],
})
print(f"({time.time() - started:.1f} s)\n")
print(follow_up["answer"])
print("Citations:", [c["source"] for c in follow_up["citations"]])

Now the expense check. Run the cell below to check every sample report through the deployed function. Each response includes the service tier that actually served the request.

In [ ]:
with open("../data/expense_reports.json") as f:
    EXPENSE_REPORTS = json.load(f)

correct = 0
for report in EXPENSE_REPORTS:
    started = time.time()
    check = invoke({"action": "check_expense", "report": report})
    correct += check["status"] == report["expected_with_policy"].lower()
    print(f"{report['id']}  {check['status']:<9} expected {report['expected_with_policy'].lower():<9} "
          f"tier: {check['service_tier']:<9} {time.time() - started:.1f} s")
    print(f"         {check['reasons'][0]}")

print(f"\n{correct}/{len(EXPENSE_REPORTS)} match the policy")

## 6. Review invocation logs <a id="s4"></a>

Every model call the function made is now in the invocation log group from Module 0, with the model, caller, token counts, service tier and `requestMetadata` tags.

Run the cell below to query the logs with [CloudWatch Logs Insights](https://docs.aws.amazon.com/AmazonCloudWatch/latest/logs/AnalyzingLogData.html) and list the function's calls. Records usually arrive within a minute, so the cell waits for them.

In [ ]:
def run_query(query, start_time=None, hours=12):
    """Run a CloudWatch Logs Insights query on the invocation logs and return the rows."""
    query_id = logs.start_query(
        logGroupName=LOG_GROUP_NAME,
        startTime=int(start_time or time.time() - hours * 3600),
        endTime=int(time.time()),
        queryString=query,
    )["queryId"]
    while (response := logs.get_query_results(queryId=query_id))["status"] in ("Scheduled", "Running"):
        time.sleep(1)
    return [{field["field"]: field["value"] for field in row} for row in response["results"]]


FUNCTION_CALLS_QUERY = f"""
fields @timestamp, modelId, requestMetadata.feature as feature, serviceTier.type as tier,
       input.inputTokenCount as input_tokens, output.outputTokenCount as output_tokens
| filter identity.arn like /{LAMBDA_ROLE_NAME}/
| sort @timestamp desc
"""

expected_calls = 2 + len(EXPENSE_REPORTS)
started = time.time()
while len(rows := run_query(FUNCTION_CALLS_QUERY, start_time=INVOKE_STARTED - 60)) < expected_calls and time.time() - started < 180:
    time.sleep(15)

print(f"{'Time':<20} {'Model':<34} {'Feature':<14} {'Tier':<9} {'In':>6} {'Out':>6}")
for row in rows:
    # A request without serviceTier is served on the Standard tier ("default").
    print(f"{row['@timestamp'][:19]:<20} {row['modelId']:<34} {row.get('feature', '-'):<14} "
          f"{row.get('tier', 'default'):<9} {row['input_tokens']:>6} {row['output_tokens']:>6}")

The calls come from the function's role, so you can tell them apart from your own notebook calls. Run the cell below to summarize every call recorded since Module 0, by caller and model.

In [ ]:
rows = run_query("""
stats count(*) as calls, sum(input.inputTokenCount) as input_tokens, sum(output.outputTokenCount) as output_tokens
  by identity.arn as caller, modelId
| sort calls desc
""")

# Combine the sessions of each role, so each user or role appears once per model.
totals = {}
for row in rows:
    if "caller" not in row or "modelId" not in row:
        continue
    caller = "/".join(row["caller"].split(":")[-1].split("/")[:2])  # For example, "assumed-role/aim317-lambda-role"
    total = totals.setdefault((caller, row["modelId"]), [0, 0, 0])
    for i, field in enumerate(["calls", "input_tokens", "output_tokens"]):
        total[i] += int(row.get(field, 0))

print(f"{'Caller':<44} {'Model':<34} {'Calls':>6} {'Input':>9} {'Output':>9}")
for (caller, model_id), (calls, input_tokens, output_tokens) in sorted(totals.items(), key=lambda t: -t[1][0]):
    print(f"{caller[:43]:<44} {model_id:<34} {calls:>6} {input_tokens:>9} {output_tokens:>9}")

## 7. Review service tier metrics <a id="s5"></a>

Amazon Bedrock also publishes [CloudWatch metrics](https://docs.aws.amazon.com/bedrock/latest/userguide/monitoring-runtime-metrics.html) in the `AWS/Bedrock` namespace, with no setup. [Service tier metrics](https://docs.aws.amazon.com/bedrock/latest/userguide/service-tiers-inference.html) are available under three dimensions:

| Dimension | Meaning |
|:--|:--|
| `ModelId` | The model or inference profile called |
| `ServiceTier` | The tier the request asked for |
| `ResolvedServiceTier` | The tier that actually served the request |

Run the cell below to see the expense model's calls and average latency per tier over the last 12 hours. Metrics can take a few minutes to appear.

In [ ]:
def tier_metrics(model_id, hours=12):
    """Return invocation count and average latency for a model, per requested and resolved tier."""
    results = []
    metrics = cloudwatch.list_metrics(
        Namespace="AWS/Bedrock", MetricName="InvocationLatency",
        Dimensions=[{"Name": "ModelId", "Value": model_id}],
    )["Metrics"]
    for metric in metrics:
        dimensions = {d["Name"]: d["Value"] for d in metric["Dimensions"]}
        if "ServiceTier" not in dimensions:
            continue
        datapoints = cloudwatch.get_metric_statistics(
            Namespace="AWS/Bedrock", MetricName="InvocationLatency", Dimensions=metric["Dimensions"],
            StartTime=time.time() - hours * 3600, EndTime=time.time(), Period=hours * 3600,
            Statistics=["Average", "SampleCount"],
        )["Datapoints"]
        if datapoints:
            results.append((dimensions["ServiceTier"], dimensions["ResolvedServiceTier"],
                            int(datapoints[0]["SampleCount"]), datapoints[0]["Average"]))
    return sorted(results)


print(f"{'Requested tier':<16} {'Resolved tier':<15} {'Calls':>6} {'Average latency':>16}")
for requested, resolved, calls, latency in tier_metrics(EXPENSE_MODEL_ID):
    print(f"{requested:<16} {resolved:<15} {calls:>6} {latency:>13.0f} ms")

`ResolvedServiceTier` shows the tier that served each request, so you can confirm that the deployed expense checks ran on `priority`.

## 8. Attribute cost with request metadata <a id="s6"></a>

The Converse [`requestMetadata`](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html) field adds key-value pairs to a request, and they appear in the invocation logs. The function tags every call with `feature` (`ask` or `check_expense`), so you can attribute tokens and cost to each feature.

Run the cell below to add up the function's tokens by feature and estimate the cost.

In [ ]:
# Example Standard tier prices, USD per 1 million tokens. For current prices, see
# https://aws.amazon.com/bedrock/pricing/
PRICES = {
    CHAT_MODEL_ID: {"input": 3.00, "output": 15.00},
    EXPENSE_MODEL_ID: {"input": 0.15, "output": 0.60},
}
# Priority is priced at a 75% premium over Standard, and Flex at a 50% discount.
TIER_MULTIPLIER = {"default": 1.0, "priority": 1.75, "flex": 0.5}

rows = run_query(f"""
filter identity.arn like /{LAMBDA_ROLE_NAME}/
| stats count(*) as calls, sum(input.inputTokenCount) as input_tokens, sum(output.outputTokenCount) as output_tokens
  by requestMetadata.feature as feature, modelId, serviceTier.type as tier
""")

print(f"{'Feature':<14} {'Model':<34} {'Tier':<9} {'Calls':>6} {'Cost':>9} {'Per 1,000':>10}")
for row in sorted(rows, key=lambda r: r.get("feature", "")):
    tier = row.get("tier", "default")
    price = PRICES[row["modelId"]]
    cost = (int(row["input_tokens"]) * price["input"] + int(row["output_tokens"]) * price["output"]) \
        * TIER_MULTIPLIER[tier] / 1e6
    per_thousand = cost / int(row["calls"]) * 1000
    print(f"{row.get('feature', '-'):<14} {row['modelId']:<34} {tier:<9} {row['calls']:>6} "
          f"{'$' + format(cost, '.4f'):>9} {'$' + format(per_thousand, '.2f'):>10}")

The estimate uses the example prices in the cell. For current prices, see the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/); your AWS bill is the authoritative record.

## 9. Try it <a id="try-it"></a>

1. Change the question in the cell below to one of your own, and run it.
2. The cell also sends an expense report with a passport number in an item description. The guardrail masks the number before the model sees it. Does the number appear in the reason?

In [ ]:
answer = invoke({"action": "ask", "question": "Can I expense a taxi from the airport to my hotel?"})
print(answer["answer"])
print("Guardrail intervened:", answer["guardrail_intervened"])

report = {
    "id": "EXP-100",
    "employee": "Your Name",
    "trip": "London, 3 nights, client visit",
    "items": [{"type": "other", "description": "Passport renewal fee, passport number 586204931", "amount": 130.0}],
}
check = invoke({"action": "check_expense", "report": report})
print(f"\n{check['status']}: {check['reasons'][0]}")
print("Passport number in the reason:", "586204931" in check["reasons"][0])

## 10. Conclusion and key takeaways <a id="conclusion"></a>

- The deployed assistant is **one stateless Lambda function** with two requests. The code decides the flow, so every request follows the same steps.
- **Invocation logs** record every model call, with the caller, tokens, service tier and tags, and CloudWatch Logs Insights queries them.
- **CloudWatch metrics** need no setup; `ResolvedServiceTier` shows the tier that served each request.
- **`requestMetadata` tags** attribute cost to the features of your application.

## 11. Next steps <a id="next-steps"></a>

You have built, deployed and monitored an application on Amazon Bedrock. The same six steps make a checklist for your next one:

| Step | Question to answer | Where you did it |
|:--|:--|:--|
| Model | Which model fits each task, by quality, accuracy, latency and cost? | Module 2 |
| Grounding | Which documents should answers come from? | Module 3 |
| Guardrails | What must the application never pass on, help with, or make up? | Module 4 |
| Service tier | Where is someone waiting, and where can work wait? | Module 5 |
| Deploy | Which requests does the application handle, and what does each call? | Module 6 |
| Monitor | What do you log, which metrics do you watch, and how do you tag calls? | Modules 0 and 6 |

From here, explore [batch inference](https://docs.aws.amazon.com/bedrock/latest/userguide/batch-inference.html) for work that can wait hours, tool use and agents, and other Managed Knowledge Base connectors. The [stretch notebook](stretch_agentcore.ipynb) turns the assistant into an agent on Amazon Bedrock AgentCore Runtime.

When you are done, run the [Cleanup](99_cleanup.ipynb) notebook to delete the resources you created.